In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# ── Load core data sources ─────────────────────────────────────────────
DATA_DIR = '/home/h604827/ControlActions/DATA'
TARGET_PV_COL = '03LIC_1071.PV'
SSD_FILE = f'{DATA_DIR}/SSD_1071_SSD_output_1071_7Jan2026.xlsx'

events_df = pd.read_csv(f'{DATA_DIR}/trip_filtered_events.csv', low_memory=False)
events_df['VT_Start'] = pd.to_datetime(events_df['VT_Start'])
events_df = events_df.sort_values('VT_Start').reset_index(drop=True)

ts_df = pd.read_parquet(f'{DATA_DIR}/03LIC_1071_JAN_2026.parquet')
ts_df['TimeStamp'] = pd.to_datetime(ts_df['TimeStamp'])
ts_df = ts_df.set_index('TimeStamp').sort_index()

op_limits = pd.read_csv(f'{DATA_DIR}/operating_limits.csv')
target_limit_row = op_limits.loc[op_limits['TAG_NAME'] == TARGET_PV_COL].squeeze()
TARGET_LOWER_LIMIT = float(target_limit_row['LOWER_LIMIT'])
TARGET_UPPER_LIMIT = float(target_limit_row['UPPER_LIMIT'])

ssd_df = pd.read_excel(SSD_FILE)
ssd_datetime_cols = [
    'AlarmStart_rounded_minutes',
    'AlarmEnd_rounded_minutes',
    'Tag_First_Transition_Start_minutes',
]
for col in ssd_datetime_cols:
    ssd_df[col] = pd.to_datetime(ssd_df[col], errors='coerce')

ssd_df = ssd_df.dropna(subset=ssd_datetime_cols).sort_values(
    ['AlarmStart_rounded_minutes', 'AlarmEnd_rounded_minutes', 'Tag_First_Transition_Start_minutes']
).reset_index(drop=True)

print(f"Events:      {len(events_df):,} rows, {events_df['VT_Start'].min().date()} to {events_df['VT_Start'].max().date()}")
print(f"Time series: {len(ts_df):,} rows, {ts_df.index.min().date()} to {ts_df.index.max().date()}")
print(f"SSD rows:    {len(ssd_df):,} rows across {ssd_df.groupby(['AlarmStart_rounded_minutes', 'AlarmEnd_rounded_minutes']).ngroups:,} alarm episodes")
print(f"Target operating limits for {TARGET_PV_COL}: {TARGET_LOWER_LIMIT:.2f} to {TARGET_UPPER_LIMIT:.2f}")

Events:      1,600,520 rows, 2021-09-06 to 2025-06-27
Time series: 1,737,586 rows, 2022-01-03 to 2025-06-23
SSD rows:    10,107 rows across 609 alarm episodes
Target operating limits for 03LIC_1071.PV: 35.25 to 42.41


In [2]:
ts_df.columns

Index(['03LIC_1071.PV', '03LIC_1071.OP', '02FI_1000.PV', '03FIC_1085.OP',
       '03FIC_1085.PV', '03FIC_3415.OP', '03FIC_3415.PV', '03FIC_3435.PV',
       '03FI_1141A.PV', '03FI_1151.PV', '03FI_3418.PV', '03LIC_1016.OP',
       '03LIC_1016.PV', '03LIC_1085.OP', '03LIC_1085.PV', '03LIC_1094.OP',
       '03LIC_1094.PV', '03LIC_1097.OP', '03LIC_1097.PV', '03LIC_3178.OP',
       '03LIC_3178.PV', '03LI_3411.PV', '03PIC_1013.OP', '03PIC_1013.PV',
       '03PIC_1068.OP', '03PIC_1068.PV', '03PIC_1104.OP', '03PIC_1104.PV',
       '03PIC_3131.OP', '03PIC_3131.PV', '03PI_1141A.PV', '03PI_1495.PV',
       '03PI_1814.PV', '03TIC_1092.OP', '03TIC_1092.PV', '03TIC_1142.OP',
       '03TIC_1142.PV', '03TIC_1145.OP', '03TIC_1145.PV', '03TI_1015.PV',
       '03TI_1081.PV', '03TI_1421.PV', '03TI_1901.PV', 'AlarmStatus',
       'AlarmType'],
      dtype='object')

In [3]:
# ── Extract OP/SP CHANGE events for tags with OP in time series ────────
# These are operator actions that change the controller output or setpoint

# Tags that have OP columns in the time series
op_cols = [c for c in ts_df.columns if c.endswith('.OP')]
op_tag_names = [c.replace('.OP', '') for c in op_cols]

# Filter CHANGE events: OP or SP changes with valid PrevValue
change_events = events_df[
    (events_df['ConditionName'] == 'CHANGE') &
    (events_df['Description'].isin(['OP', 'SP'])) &
    (events_df['Source'].isin(op_tag_names))
].copy()

# Remove duplicates (same timestamp, keep the one with PrevValue)
dup_mask = change_events.duplicated(subset=['VT_Start', 'Source', 'Description'], keep=False)
change_events = change_events[~(dup_mask & change_events['PrevValue'].isna())]

# Convert to numeric
change_events['PrevValue'] = pd.to_numeric(change_events['PrevValue'], errors='coerce')
change_events['Value'] = pd.to_numeric(change_events['Value'], errors='coerce')
change_events = change_events.dropna(subset=['PrevValue', 'Value'])

# Compute change magnitude
change_events['delta'] = change_events['Value'] - change_events['PrevValue']
change_events['abs_delta'] = change_events['delta'].abs()

# Drop trivial changes (delta == 0)
change_events = change_events[change_events['abs_delta'] > 0].copy()

print(f"Total OP/SP change events (non-zero delta): {len(change_events):,}")
print(f"\nPer-tag breakdown:")
per_tag_counts = (
    change_events
    .groupby(['Source', 'Description'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['OP', 'SP'], fill_value=0)
    .rename(columns={'OP': 'op_actions', 'SP': 'sp_actions'})
)

per_tag_counts['total_changes'] = per_tag_counts['op_actions'] + per_tag_counts['sp_actions']
per_tag_counts = per_tag_counts.sort_values('total_changes', ascending=False)

print(per_tag_counts.to_string())

Total OP/SP change events (non-zero delta): 23,750

Per-tag breakdown:
Description  op_actions  sp_actions  total_changes
Source                                            
03PIC_1013         7390           0           7390
03FIC_3415         5555          20           5575
03LIC_1085            5        2954           2959
03PIC_3131         1445         389           1834
03LIC_1071          939         462           1401
03LIC_1097           13         892            905
03LIC_1094          753         136            889
03FIC_1085          837           0            837
03LIC_1016          435         321            756
03LIC_3178          351         333            684
03PIC_1068           46         435            481
03PIC_1104           27           7             34
03TIC_1092            0           5              5


# Response Time And Settling Analysis: Alarm-Episode OP Actions To 03LIC_1071.PV

## Scope
This notebook now focuses on OP and SP control actions taken during alarm episodes, using SSD-derived episode boundaries:

1. Use SSD to identify each alarm episode from the earliest transition start to alarm end.
2. Extend each episode window to `alarm end + 60 minutes`.
3. Collect all OP and SP change events that fall inside those windows.
4. Measure how `03LIC_1071.PV` responded after each action.

## Questions Answered
1. Which tags were adjusted during alarm episodes, and what is the OP versus SP action distribution?
2. What was `03LIC_1071.PV` when each action was taken?
3. How long did it take for `03LIC_1071.PV` to respond?
4. How long did it take to settle after the action?
5. What value did `03LIC_1071.PV` settle around, and was that inside or outside its operating limits?

## Method
- Build one episode window per SSD alarm episode using the earliest `Tag_First_Transition_Start_minutes` and `AlarmEnd_rounded_minutes + 60 minutes`.
- Keep every OP or SP change event that falls inside any episode window.
- For each action, measure settling only until the next action in the same episode or the episode window end, whichever comes first.
- Measure baseline variability from the 10 minutes before the action.
- Measure response time from the first full minute after the action.
- Define settling time as the first sustained period where rolling PV standard deviation falls below 50% of the baseline standard deviation.

In [4]:
# ── Build SSD alarm episode windows and collect all actions inside them ───
POST_ALARM_BUFFER_MINS = 60

# Create one row per alarm episode using the earliest SSD transition start
alarm_windows = (
    ssd_df
    .groupby(['AlarmStart_rounded_minutes', 'AlarmEnd_rounded_minutes'], as_index=False)
    .agg(EarliestTransitionStart=('Tag_First_Transition_Start_minutes', 'min'))
    .sort_values('AlarmStart_rounded_minutes')
    .reset_index(drop=True)
)

alarm_windows['EpisodeID'] = np.arange(1, len(alarm_windows) + 1)
alarm_windows['window_start'] = alarm_windows['EarliestTransitionStart']
alarm_windows['window_end'] = alarm_windows['AlarmEnd_rounded_minutes'] + pd.Timedelta(minutes=POST_ALARM_BUFFER_MINS)
alarm_windows['TransitionToAlarmMinutes'] = (
    (alarm_windows['AlarmStart_rounded_minutes'] - alarm_windows['window_start']).dt.total_seconds() / 60
)
alarm_windows['AlarmDurationMinutes'] = (
    (alarm_windows['AlarmEnd_rounded_minutes'] - alarm_windows['AlarmStart_rounded_minutes']).dt.total_seconds() / 60
)
alarm_windows['WindowDurationMinutes'] = (
    (alarm_windows['window_end'] - alarm_windows['window_start']).dt.total_seconds() / 60
)

# Cross-join change events to episode windows and keep events that fall inside the window.
# This is simple and reliable for the current episode count.
episode_actions = (
    change_events.assign(_merge_key=1)
    .merge(alarm_windows.assign(_merge_key=1), on='_merge_key', how='inner')
    .drop(columns='_merge_key')
)

episode_actions = episode_actions[
    (episode_actions['VT_Start'] >= episode_actions['window_start']) &
    (episode_actions['VT_Start'] <= episode_actions['window_end'])
].copy()

episode_actions = episode_actions.sort_values(['EpisodeID', 'VT_Start', 'Source', 'Description']).reset_index(drop=True)
episode_actions['action_rank_in_episode'] = episode_actions.groupby('EpisodeID').cumcount() + 1
episode_actions['next_action_time'] = episode_actions.groupby('EpisodeID')['VT_Start'].shift(-1)

episode_action_distribution = (
    episode_actions
    .groupby(['Source', 'Description'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['OP', 'SP'], fill_value=0)
    .rename(columns={'OP': 'op_actions', 'SP': 'sp_actions'})
)
episode_action_distribution['total_changes'] = (
    episode_action_distribution['op_actions'] + episode_action_distribution['sp_actions']
)
episode_action_distribution = episode_action_distribution.sort_values('total_changes', ascending=False)

print(f"Alarm episodes from SSD: {len(alarm_windows):,}")
print(f"Episode actions in [transition start, alarm end + {POST_ALARM_BUFFER_MINS} min]: {len(episode_actions):,}")
print(f"Episodes with at least one action: {episode_actions['EpisodeID'].nunique():,}")
print()
print(alarm_windows[['EpisodeID', 'AlarmStart_rounded_minutes', 'AlarmEnd_rounded_minutes', 'window_start', 'window_end']].head().to_string(index=False))
print()
print('Per-tag OP/SP distribution within episode windows:')
print(episode_action_distribution.to_string())

Alarm episodes from SSD: 609
Episode actions in [transition start, alarm end + 60 min]: 7,744
Episodes with at least one action: 303

 EpisodeID AlarmStart_rounded_minutes AlarmEnd_rounded_minutes        window_start          window_end
         1        2022-01-05 08:53:00      2022-01-05 09:33:00 2022-01-05 07:27:00 2022-01-05 10:33:00
         2        2022-01-07 09:55:00      2022-01-07 10:00:00 2022-01-07 08:29:00 2022-01-07 11:00:00
         3        2022-01-07 13:33:00      2022-01-07 13:36:00 2022-01-07 12:07:00 2022-01-07 14:36:00
         4        2022-01-07 14:17:00      2022-01-07 14:19:00 2022-01-07 12:51:00 2022-01-07 15:19:00
         5        2022-01-07 14:54:00      2022-01-07 14:58:00 2022-01-07 13:28:00 2022-01-07 15:58:00

Per-tag OP/SP distribution within episode windows:
Description  op_actions  sp_actions  total_changes
Source                                            
03PIC_1013         3183           0           3183
03LIC_1071         1221         393        

In [5]:
# ── Measure response time and settling outcome for each episode action ─────
PRE_WINDOW = 10
POST_WINDOW = 60
RESPONSE_THRESHOLD_FACTOR = 0.5
SETTLING_STD_WINDOW = 5
SETTLING_STD_FACTOR = 0.5
SETTLED_VALUE_WINDOW = 5
MIN_ACTION_GAP_FOR_SETTLING = 5

def classify_limit_zone(value, lower_limit, upper_limit):
    if pd.isna(value):
        return 'unknown'
    if value < lower_limit:
        return 'below_lower_limit'
    if value > upper_limit:
        return 'above_upper_limit'
    return 'within_limits'

def distance_outside_limits(value, lower_limit, upper_limit):
    if pd.isna(value):
        return np.nan
    if value < lower_limit:
        return lower_limit - value
    if value > upper_limit:
        return value - upper_limit
    return 0.0

results = []

for _, row in episode_actions.iterrows():
    t0 = row['VT_Start']
    tag = row['Source']
    desc = row['Description']
    op_delta = row['delta']
    t0_rounded = t0.floor('min')

    measurement_end = min(
        t0_rounded + timedelta(minutes=POST_WINDOW),
        row['window_end']
    )
    if pd.notna(row['next_action_time']):
        next_action_floor = row['next_action_time'].floor('min')
        measurement_end = min(measurement_end, next_action_floor)

    pre_start = t0_rounded - timedelta(minutes=PRE_WINDOW)
    pre_end = t0_rounded

    baseline = ts_df.loc[pre_start:pre_end, TARGET_PV_COL].dropna()
    post_pv = ts_df.loc[t0_rounded:measurement_end, TARGET_PV_COL].dropna()

    if len(baseline) < 3 or len(post_pv) < 2:
        continue

    baseline_mean = baseline.mean()
    baseline_std = baseline.std()
    if pd.isna(baseline_std) or baseline_std < 0.1:
        baseline_std = 0.1

    pv_at_action = post_pv.iloc[0]
    post_action_pv = post_pv.iloc[1:].copy()
    available_post_minutes = max((measurement_end - t0_rounded).total_seconds() / 60, 0)

    response_time_mins = np.nan
    settling_time_mins = np.nan
    settled_pv = np.nan
    settled_pv_std = np.nan
    settled_start_time = pd.NaT
    settled_end_time = pd.NaT
    settlement_observation_end = measurement_end
    settling_status = 'insufficient_post_window'

    if len(post_action_pv) >= 1:
        response_threshold = RESPONSE_THRESHOLD_FACTOR * baseline_std
        pv_deviation = (post_action_pv - baseline_mean).abs()
        response_candidates = pv_deviation[pv_deviation > response_threshold].index
        if len(response_candidates) > 0:
            response_time_mins = (response_candidates[0] - t0_rounded).total_seconds() / 60

    if available_post_minutes >= max(SETTLING_STD_WINDOW, MIN_ACTION_GAP_FOR_SETTLING) and len(post_action_pv) >= SETTLING_STD_WINDOW:
        rolling_std = post_action_pv.rolling(window=SETTLING_STD_WINDOW, min_periods=3).std()
        settle_threshold = SETTLING_STD_FACTOR * baseline_std
        settled_mask = (rolling_std < settle_threshold).fillna(False)
        settled_groups = settled_mask.astype(int).groupby((settled_mask.astype(int) != settled_mask.astype(int).shift()).cumsum())

        for _, grp in settled_groups:
            if grp.iloc[0] != 1 or len(grp) < SETTLED_VALUE_WINDOW:
                continue
            settled_start_time = grp.index[0]
            settled_end_time = grp.index[SETTLED_VALUE_WINDOW - 1]
            settled_slice = post_action_pv.loc[settled_start_time:settled_end_time].dropna()
            if len(settled_slice) < 3:
                continue
            settling_time_mins = (settled_start_time - t0_rounded).total_seconds() / 60
            settled_pv = settled_slice.mean()
            settled_pv_std = settled_slice.std()
            settling_status = 'settled'
            break

        if settling_status != 'settled':
            settling_status = 'no_settled_segment_before_next_action'

    action_zone = classify_limit_zone(pv_at_action, TARGET_LOWER_LIMIT, TARGET_UPPER_LIMIT)
    settled_zone = classify_limit_zone(settled_pv, TARGET_LOWER_LIMIT, TARGET_UPPER_LIMIT)
    action_outside_distance = distance_outside_limits(pv_at_action, TARGET_LOWER_LIMIT, TARGET_UPPER_LIMIT)
    settled_outside_distance = distance_outside_limits(settled_pv, TARGET_LOWER_LIMIT, TARGET_UPPER_LIMIT)
    movement_towards_limits = action_outside_distance - settled_outside_distance if pd.notna(settled_outside_distance) else np.nan

    op_col = f'{tag}.OP'
    op_value_at_change = np.nan
    if op_col in ts_df.columns and t0_rounded in ts_df.index:
        op_value_at_change = ts_df.at[t0_rounded, op_col]

    results.append({
        'EpisodeID': row['EpisodeID'],
        'AlarmStart_rounded_minutes': row['AlarmStart_rounded_minutes'],
        'AlarmEnd_rounded_minutes': row['AlarmEnd_rounded_minutes'],
        'window_start': row['window_start'],
        'window_end': row['window_end'],
        'action_rank_in_episode': row['action_rank_in_episode'],
        'next_action_time': row['next_action_time'],
        'measurement_end': measurement_end,
        'settlement_observation_end': settlement_observation_end,
        'event_time': t0,
        'event_time_rounded': t0_rounded,
        'source_tag': tag,
        'change_type': desc,
        'op_delta': op_delta,
        'abs_op_delta': abs(op_delta),
        'action_direction': 'increase' if op_delta > 0 else 'decrease',
        'op_value_at_change': op_value_at_change,
        'pv_baseline_mean': baseline_mean,
        'pv_baseline_std': baseline_std,
        'pv_at_action': pv_at_action,
        'action_zone': action_zone,
        'action_outside_distance': action_outside_distance,
        'response_time_mins': response_time_mins,
        'settling_time_mins': settling_time_mins,
        'settling_status': settling_status,
        'available_post_minutes': available_post_minutes,
        'settled_pv': settled_pv,
        'settled_pv_std': settled_pv_std,
        'settled_start_time': settled_start_time,
        'settled_end_time': settled_end_time,
        'settled_zone': settled_zone,
        'settled_outside_distance': settled_outside_distance,
        'movement_towards_limits': movement_towards_limits,
        'moved_closer_to_limits': movement_towards_limits > 0 if pd.notna(movement_towards_limits) else False,
        'pv_change_to_settled': settled_pv - pv_at_action if pd.notna(settled_pv) else np.nan,
        'year': t0.year,
    })

response_df = pd.DataFrame(results)
response_df['settled_within_limits'] = response_df['settled_zone'] == 'within_limits'

print(f"Computed metrics for {len(response_df):,} episode-window actions")
print(f"Actions with measurable response: {response_df['response_time_mins'].notna().sum():,}")
print(f"Actions that settled before the next action or window end: {response_df['settling_time_mins'].notna().sum():,}")
print()
print('Action direction split:')
print(response_df['action_direction'].value_counts().to_string())
print()
print('Settling status split:')
print(response_df['settling_status'].value_counts(dropna=False).to_string())
print()
print('Settled zone split:')
print(response_df['settled_zone'].value_counts(dropna=False).to_string())

Computed metrics for 3,071 episode-window actions
Actions with measurable response: 2,752
Actions that settled before the next action or window end: 479

Action direction split:
action_direction
decrease    1583
increase    1488

Settling status split:
settling_status
insufficient_post_window                 1896
no_settled_segment_before_next_action     696
settled                                   479

Settled zone split:
settled_zone
unknown              2592
below_lower_limit     239
within_limits         124
above_upper_limit     116


## Episode-Window Outputs

The outputs below are intentionally limited to the operational questions for actions taken during alarm episodes:

- Which tag was moved, and was it an OP or SP action?
- What was the action direction and magnitude?
- What was `03LIC_1071.PV` at the time of the action?
- How long did it take to respond and settle before the next intervention?
- What value did it settle around?
- Did it settle back inside the operating limits?

In [6]:
# ── Summary tables for episode-window action review ─────────────────────
print('OP/SP action distribution by tag during alarm episode windows:')
display(episode_action_distribution.reset_index().rename(columns={'Source': 'source_tag'}))

summary_df = response_df.groupby(['source_tag', 'change_type', 'action_direction']).agg(
    n_actions=('event_time', 'count'),
    n_episodes=('EpisodeID', 'nunique'),
    min_action_change=('abs_op_delta', 'min'),
    median_action_change=('abs_op_delta', 'median'),
    mean_action_change=('abs_op_delta', 'mean'),
    max_action_change=('abs_op_delta', 'max'),
    median_response_time_min=('response_time_mins', 'median'),
    median_settling_time_min=('settling_time_mins', 'median'),
    median_pv_at_action=('pv_at_action', 'median'),
    median_settled_pv=('settled_pv', 'median'),
    pct_settled_within_limits=('settled_within_limits', lambda s: 100 * s.mean()),
    pct_actions_settled=('settling_time_mins', lambda s: 100 * s.notna().mean()),
    median_limit_improvement=('movement_towards_limits', 'median'),
).reset_index()

summary_df = summary_df.sort_values(
    ['n_actions', 'source_tag', 'change_type', 'action_direction'],
    ascending=[False, True, True, True]
)
display(summary_df.round(2))

event_review_cols = [
    'EpisodeID', 'event_time', 'source_tag', 'change_type', 'action_direction', 'op_delta',
    'pv_at_action', 'action_zone', 'response_time_mins', 'settling_time_mins',
    'settling_status', 'settled_pv', 'settled_zone', 'movement_towards_limits'
]
event_review_df = response_df[event_review_cols].sort_values(['EpisodeID', 'event_time'], ascending=[False, False])
display(event_review_df.head(30).round(2))

OP/SP action distribution by tag during alarm episode windows:


Description,source_tag,op_actions,sp_actions,total_changes
0,03PIC_1013,3183,0,3183
1,03LIC_1071,1221,393,1614
2,03PIC_3131,386,400,786
3,03FIC_3415,507,0,507
4,03LIC_1016,272,205,477
5,03LIC_1085,0,401,401
6,03FIC_1085,330,0,330
7,03PIC_1068,9,169,178
8,03LIC_3178,105,48,153
9,03LIC_1097,7,61,68


,source_tag,change_type,action_direction,n_actions,n_episodes,min_action_change,median_action_change,mean_action_change,max_action_change,median_response_time_min,median_settling_time_min,median_pv_at_action,median_settled_pv,pct_settled_within_limits,pct_actions_settled,median_limit_improvement
26,03PIC_1013,OP,decrease,651,126,0.08,2.00,1.37,4.00,1.0,9.0,37.82,38.99,4.76,16.13,0.07
27,03PIC_1013,OP,increase,418,82,0.10,2.00,1.46,4.00,1.0,7.5,39.08,36.16,4.31,11.00,0.00
9,03LIC_1071,OP,increase,309,41,0.10,2.00,3.26,35.00,1.0,3.0,29.80,26.02,0.00,15.21,0.00
8,03LIC_1071,OP,decrease,245,42,0.10,2.00,4.70,50.00,1.0,3.0,47.65,33.29,0.00,8.57,-0.00
12,03LIC_1085,SP,decrease,173,51,0.10,1.00,1.16,8.81,1.0,3.0,35.40,34.60,5.20,19.65,0.00
11,03LIC_1071,SP,increase,145,68,0.01,2.00,2.56,35.00,1.0,7.0,29.29,37.98,11.03,31.72,0.35
2,03FIC_3415,OP,decrease,121,37,1.10,5.00,6.63,70.00,1.0,3.0,34.68,34.90,9.09,25.62,0.81
3,03FIC_3415,OP,increase,101,36,2.00,5.00,6.89,35.00,1.0,4.5,37.53,36.51,12.87,19.80,0.00
1,03FIC_1085,OP,increase,88,25,0.10,2.00,3.17,17.97,1.0,3.0,35.00,-0.23,0.00,2.27,-0.71
7,03LIC_1016,SP,increase,81,33,0.30,2.00,2.46,15.00,1.0,8.0,28.75,39.95,7.41,20.99,0.00


,EpisodeID,event_time,source_tag,change_type,action_direction,op_delta,pv_at_action,action_zone,response_time_mins,settling_time_mins,settling_status,settled_pv,settled_zone,movement_towards_limits
3070,609,2025-06-22 17:29:11.555400,03LIC_1071,SP,increase,1.0,44.31,above_upper_limit,2.0,NaN,no_settled_segment_before_next_action,NaN,unknown,NaN
3069,609,2025-06-22 17:27:43.206500,03LIC_1071,SP,increase,2.0,31.69,below_lower_limit,2.0,NaN,insufficient_post_window,NaN,unknown,NaN
3068,609,2025-06-22 17:03:51.553700,03PIC_1013,OP,decrease,-2.0,45.96,above_upper_limit,1.0,NaN,no_settled_segment_before_next_action,NaN,unknown,NaN
3067,609,2025-06-22 16:29:10.484100,03LIC_1085,SP,increase,0.2,32.46,below_lower_limit,1.0,17.0,settled,42.29,within_limits,2.79
3066,609,2025-06-22 16:21:45.382200,03PIC_1013,OP,increase,2.0,31.72,below_lower_limit,1.0,NaN,no_settled_segment_before_next_action,NaN,unknown,NaN
3065,609,2025-06-22 16:19:45.446300,03PIC_1013,OP,increase,2.0,45.77,above_upper_limit,1.0,NaN,insufficient_post_window,NaN,unknown,NaN
3064,609,2025-06-22 16:04:26.492700,03LIC_1071,SP,increase,2.0,32.39,below_lower_limit,1.0,NaN,no_settled_segment_before_next_action,NaN,unknown,NaN
3063,608,2025-06-22 17:03:51.553700,03PIC_1013,OP,decrease,-2.0,45.96,above_upper_limit,1.0,NaN,insufficient_post_window,NaN,unknown,NaN
3062,608,2025-06-22 16:29:10.484100,03LIC_1085,SP,increase,0.2,32.46,below_lower_limit,1.0,17.0,settled,42.29,within_limits,2.79
3061,608,2025-06-22 16:21:45.382200,03PIC_1013,OP,increase,2.0,31.72,below_lower_limit,1.0,NaN,no_settled_segment_before_next_action,NaN,unknown,NaN


In [7]:
# ── Visual 1: Settling-time histogram with action-magnitude stats ────────
hist_df = response_df.dropna(subset=['settling_time_mins', 'abs_op_delta']).copy()
tag_counts = hist_df['source_tag'].value_counts()
hist_tags = tag_counts[tag_counts >= 10].index.tolist()
hist_df = hist_df[hist_df['source_tag'].isin(hist_tags)].copy()

BIN_WIDTH = 5
max_settling = hist_df['settling_time_mins'].max()
bin_edges = np.arange(0, np.ceil(max_settling / BIN_WIDTH) * BIN_WIDTH + BIN_WIDTH, BIN_WIDTH)
hist_df['settling_bin'] = pd.cut(
    hist_df['settling_time_mins'],
    bins=bin_edges,
    right=False,
    include_lowest=True,
)
hist_df = hist_df.dropna(subset=['settling_bin']).copy()
hist_df['bin_start'] = hist_df['settling_bin'].map(lambda interval: float(interval.left)).astype(float)
hist_df['bin_end'] = hist_df['bin_start'] + BIN_WIDTH
hist_df['bin_label'] = hist_df.apply(
    lambda row: f"{int(row['bin_start'])}-{int(row['bin_end'])} min", axis=1
)

hist_summary = hist_df.groupby([
    'source_tag', 'change_type', 'action_direction', 'bin_start', 'bin_end', 'bin_label'
]).agg(
    n_actions=('event_time', 'count'),
    median_settling_time=('settling_time_mins', 'median'),
    min_action_change=('abs_op_delta', 'min'),
    median_action_change=('abs_op_delta', 'median'),
    mean_action_change=('abs_op_delta', 'mean'),
    max_action_change=('abs_op_delta', 'max'),
).reset_index()

hist_summary['panel'] = (
    hist_summary['source_tag']
    + ' | '
    + hist_summary['change_type']
    + ' | '
    + hist_summary['action_direction'].str.title()
)
panel_order = []
for tag in hist_tags:
    for change_type in ['OP', 'SP']:
        for direction in ['increase', 'decrease']:
            panel_name = f'{tag} | {change_type} | {direction.title()}'
            if panel_name in hist_summary['panel'].values:
                panel_order.append(panel_name)

fig = px.bar(
    hist_summary,
    x='bin_start',
    y='n_actions',
    facet_col='panel',
    facet_col_wrap=4,
    category_orders={'panel': panel_order},
    custom_data=[
        'source_tag', 'change_type', 'action_direction', 'bin_label', 'n_actions',
        'min_action_change', 'median_action_change', 'mean_action_change', 'max_action_change',
        'median_settling_time'
    ],
    title='Settling-Time Histogram For Alarm-Episode Actions',
    labels={
        'bin_start': 'Settling-time bin start (minutes)',
        'n_actions': 'Number of actions',
        'panel': 'Tag | action type | direction'
    }
)

fig.update_traces(
    hovertemplate=(
        'Tag: %{customdata[0]}<br>'
        'Action type: %{customdata[1]}<br>'
        'Direction: %{customdata[2]}<br>'
        'Settling-time bin: %{customdata[3]}<br>'
        'Actions in bin: %{customdata[4]}<br>'
        'Median settling time: %{customdata[9]:.1f} min<br>'
        'Min |change|: %{customdata[5]:.2f}<br>'
        'Median |change|: %{customdata[6]:.2f}<br>'
        'Mean |change|: %{customdata[7]:.2f}<br>'
        'Max |change|: %{customdata[8]:.2f}<extra></extra>'
    )
)

fig.update_xaxes(
    tickmode='linear',
    dtick=BIN_WIDTH,
    title_text='Settling time (minutes)'
)
fig.update_yaxes(title_text='Number of actions')
fig.update_layout(height=max(600, 260 * int(np.ceil(max(len(panel_order), 1) / 4))))
fig.show()

In [8]:

# ── Visual 4: 03LIC_1071 actions in 3D space ───────────────────────────
plot_df = response_df[response_df['source_tag'] == '03LIC_1071'].dropna(
    subset=['settling_time_mins', 'pv_at_action', 'settled_pv']
).copy()

plot_df['action_label'] = plot_df['action_direction'].str.title()
plot_df['event_label'] = plot_df['event_time'].dt.strftime('%Y-%m-%d %H:%M')

fig = px.scatter_3d(
    plot_df,
    x='settling_time_mins',
    y='pv_at_action',
    z='settled_pv',
    color='action_label',
    symbol='change_type',
    size='abs_op_delta',
    hover_data=[
        'event_label',
        'change_type',
        'op_delta',
        'abs_op_delta',
        'response_time_mins',
        'action_zone',
        'settled_zone',
        'movement_towards_limits'
    ],
    title='03LIC_1071 Actions In 3D: Settling Time vs PV At Action vs Settled PV',
    labels={
        'settling_time_mins': 'Settling time (minutes)',
        'pv_at_action': '03LIC_1071.PV when action was taken',
        'settled_pv': '03LIC_1071.PV settled around',
        'action_label': 'Action direction',
        'change_type': 'Action type',
        'abs_op_delta': '|OP change|'
    },
    color_discrete_map={'Increase': '#1f77b4', 'Decrease': '#d62728'},
    symbol_map={'OP': 'circle', 'SP': 'diamond'}
)

# ── Operating-limit & alarm-limit planes ────────────────────────────────
ALARM_LOW_LIMIT = 28.75

x_range = [plot_df['settling_time_mins'].min(), plot_df['settling_time_mins'].max()]
y_range = [plot_df['pv_at_action'].min() - 1, plot_df['pv_at_action'].max() + 1]
z_range = [plot_df['settled_pv'].min() - 1, plot_df['settled_pv'].max() + 1]

# Plane grid resolution
xx = np.linspace(x_range[0], x_range[1], 2)
yy_full = np.linspace(y_range[0], y_range[1], 2)
zz_full = np.linspace(z_range[0], z_range[1], 2)

# --- Planes along pv_at_action axis (y = constant) ---
for limit_val, color, name in [
    (ALARM_LOW_LIMIT, 'red', 'Alarm limit (PV at action)'),
    (TARGET_LOWER_LIMIT, 'firebrick', 'Lower op limit (PV at action)'),
    (TARGET_UPPER_LIMIT, 'yellow', 'Upper op limit (PV at action)'),
]:
    xx_grid, zz_grid = np.meshgrid(xx, zz_full)
    yy_grid = np.full_like(xx_grid, limit_val)
    fig.add_trace(go.Surface(
        x=xx_grid, y=yy_grid, z=zz_grid,
        colorscale=[[0, color], [1, color]],
        opacity=0.2,
        showscale=False,
        name=name,
        showlegend=True,
        hoverinfo='name',
    ))

# --- Planes along settled_pv axis (z = constant) ---
for limit_val, color, name in [
    (ALARM_LOW_LIMIT, 'red', 'Alarm limit (Settled PV)'),
    (TARGET_LOWER_LIMIT, 'firebrick', 'Lower op limit (Settled PV)'),
    (TARGET_UPPER_LIMIT, 'yellow', 'Upper op limit (Settled PV)'),
]:
    xx_grid, yy_grid = np.meshgrid(xx, yy_full)
    zz_grid = np.full_like(xx_grid, limit_val)
    fig.add_trace(go.Surface(
        x=xx_grid, y=yy_grid, z=zz_grid,
        colorscale=[[0, color], [1, color]],
        opacity=0.2,
        showscale=False,
        name=name,
        showlegend=True,
        hoverinfo='name',
    ))

fig.update_layout(
    height=750,
    scene=dict(
        xaxis_title='Settling time (minutes)',
        yaxis_title='03LIC_1071.PV at action',
        zaxis_title='03LIC_1071.PV settled',
        camera=dict(
            eye=dict(x=1.0, y=1.0, z=1.0),  # closer than default (1.25, 1.25, 1.25)
        ),
    ),
)
fig.show(config={'scrollZoom': True})


In [9]:
# ── Visual 2: 03LIC_1071 settling time vs PV at action time ─────────────
plot_df = response_df[
    (response_df['source_tag'] == '03LIC_1071') &
    response_df['settling_time_mins'].notna() &
    response_df['pv_at_action'].notna()
].copy()

plot_df['action_label'] = plot_df['action_direction'].str.title()
plot_df['event_label'] = plot_df['event_time'].dt.strftime('%Y-%m-%d %H:%M')
plot_df['episode_label'] = plot_df['EpisodeID'].apply(lambda value: f'Episode {value:03d}')

fig = px.scatter(
    plot_df,
    x='pv_at_action',
    y='settling_time_mins',
    color='action_label',
    symbol='change_type',
    size='abs_op_delta',
    hover_data=[
        'episode_label',
        'event_label',
        'change_type',
        'op_delta',
        'settled_pv',
        'response_time_mins',
        'action_zone',
        'settled_zone',
        'movement_towards_limits'
    ],
    title='03LIC_1071 Actions During Alarm Episodes: Settling Time vs 03LIC_1071.PV At Action',
    labels={
        'pv_at_action': '03LIC_1071.PV when action was taken',
        'settling_time_mins': 'Settling time (minutes)',
        'action_label': 'Action direction',
        'change_type': 'Action type',
        'abs_op_delta': '|action change|'
    },
    color_discrete_map={'Increase': '#1f77b4', 'Decrease': '#d62728'},
    symbol_map={'OP': 'circle', 'SP': 'diamond'}
)

fig.add_vline(x=TARGET_LOWER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_vline(x=TARGET_UPPER_LIMIT, line_dash='dot', line_color='darkgreen')
fig.add_vline(x=28.75, line_dash='dash', line_color='red')

fig.update_layout(height=650)
fig.show()

In [10]:
# ── Extract 03LIC_1071 episode actions near the alarm limit that recovered higher ──
ALARM_LOW_LIMIT = 28.75
AROUND_ALARM_BUFFER = 2.0
SETTLED_HIGH_PV_THRESHOLD = TARGET_LOWER_LIMIT
MIN_RECOVERY_DELTA = 3.0

recovery_actions_1071 = response_df[
    (response_df['source_tag'] == '03LIC_1071') &
    (response_df['pv_at_action'] <= ALARM_LOW_LIMIT + AROUND_ALARM_BUFFER) &
    (response_df['settled_pv'] >= SETTLED_HIGH_PV_THRESHOLD) &
    ((response_df['settled_pv'] - response_df['pv_at_action']) >= MIN_RECOVERY_DELTA) &
    response_df['settling_time_mins'].notna()
].copy()

recovery_actions_1071 = recovery_actions_1071.sort_values(
    ['pv_at_action', 'settled_pv', 'settling_time_mins'],
    ascending=[True, False, True]
).reset_index(drop=True)

recovery_actions_1071['pv_recovery'] = recovery_actions_1071['settled_pv'] - recovery_actions_1071['pv_at_action']

print('03LIC_1071 episode-action recovery filter:')
print(f'  Alarm low limit: {ALARM_LOW_LIMIT:.2f}')
print(f'  Around-alarm buffer: {AROUND_ALARM_BUFFER:.2f}')
print(f'  Settled-high threshold: {SETTLED_HIGH_PV_THRESHOLD:.2f}')
print(f'  Minimum PV recovery: {MIN_RECOVERY_DELTA:.2f}')
print()
print(f'Matching actions: {len(recovery_actions_1071)}')
print()

recovery_cols = [
    'EpisodeID',
    'event_time',
    'change_type',
    'action_direction',
    'op_delta',
    'abs_op_delta',
    'pv_at_action',
    'response_time_mins',
    'settling_time_mins',
    'settled_pv',
    'pv_recovery',
    'action_zone',
    'settled_zone',
    'movement_towards_limits',
]

display(recovery_actions_1071[recovery_cols].round(2))

if len(recovery_actions_1071) > 0:
    fig = px.scatter(
        recovery_actions_1071,
        x='pv_at_action',
        y='settled_pv',
        color='action_direction',
        symbol='change_type',
        size='abs_op_delta',
        hover_data=['EpisodeID', 'event_time', 'op_delta', 'pv_recovery', 'settling_time_mins'],
        title='03LIC_1071 Episode Actions: Near Alarm At Action Time, Higher PV After Settling',
        labels={
            'pv_at_action': '03LIC_1071.PV when action was taken',
            'settled_pv': '03LIC_1071.PV settled around',
            'abs_op_delta': '|action change|'
        }
    )
    fig.add_vline(x=ALARM_LOW_LIMIT, line_dash='dot', line_color='red')
    fig.add_vline(x=ALARM_LOW_LIMIT + AROUND_ALARM_BUFFER, line_dash='dash', line_color='darkred')
    fig.add_hline(y=SETTLED_HIGH_PV_THRESHOLD, line_dash='dot', line_color='darkgreen')
    fig.update_layout(height=500)
    fig.show()
else:
    print('No 03LIC_1071 episode actions matched the current recovery filter.')

03LIC_1071 episode-action recovery filter:
  Alarm low limit: 28.75
  Around-alarm buffer: 2.00
  Settled-high threshold: 35.25
  Minimum PV recovery: 3.00

Matching actions: 19



,EpisodeID,event_time,change_type,action_direction,op_delta,abs_op_delta,pv_at_action,response_time_mins,settling_time_mins,settled_pv,pv_recovery,action_zone,settled_zone,movement_towards_limits
0,603,2025-06-21 20:12:21.165800,SP,increase,5.00,5.00,9.08,1.0,22.0,39.63,30.55,below_lower_limit,within_limits,26.16
1,521,2025-01-05 08:03:02.807000,SP,increase,4.00,4.00,23.08,1.0,7.0,40.95,17.87,below_lower_limit,within_limits,12.16
2,522,2025-01-05 08:03:02.807000,SP,increase,4.00,4.00,23.08,1.0,7.0,40.95,17.87,below_lower_limit,within_limits,12.16
3,508,2024-11-30 08:25:13.253200,SP,increase,1.00,1.00,25.88,1.0,9.0,54.27,28.39,below_lower_limit,above_upper_limit,-2.48
4,509,2024-11-30 08:25:13.253200,SP,increase,1.00,1.00,25.88,1.0,9.0,54.27,28.39,below_lower_limit,above_upper_limit,-2.48
5,52,2022-03-25 23:06:00.056200,SP,increase,2.00,2.00,27.65,1.0,11.0,40.94,13.28,below_lower_limit,within_limits,7.59
6,316,2024-04-16 09:26:07.753500,SP,increase,1.21,1.21,27.92,1.0,7.0,35.72,7.80,below_lower_limit,within_limits,7.33
7,280,2024-01-27 06:49:09.105500,SP,increase,2.00,2.00,27.96,1.0,21.0,50.21,22.25,below_lower_limit,above_upper_limit,-0.51
8,281,2024-01-27 06:49:09.105500,SP,increase,2.00,2.00,27.96,1.0,21.0,50.21,22.25,below_lower_limit,above_upper_limit,-0.51
9,488,2024-11-28 19:04:58.530100,SP,increase,1.00,1.00,28.01,1.0,12.0,60.22,32.22,below_lower_limit,above_upper_limit,-10.57


## Extract 03LIC_1071 Episode Actions Near The Alarm Limit

This section pulls only the `03LIC_1071` control actions found inside alarm-episode windows where the target PV was near or below the low alarm limit when the action was taken, but later settled back to materially higher values.

The thresholds are explicit and can be tuned:

- `ALARM_LOW_LIMIT`: low alarm threshold for `03LIC_1071.PV`
- `AROUND_ALARM_BUFFER`: how far above the alarm threshold still counts as "around alarm limit"
- `SETTLED_HIGH_PV_THRESHOLD`: minimum settled PV considered a strong recovery
- `MIN_RECOVERY_DELTA`: minimum increase from action-time PV to settled PV

In [11]:
# ── Visual 3: 03LIC_1071 action-time PV vs settled PV ───────────────────
plot_df = response_df[response_df['source_tag'] == '03LIC_1071'].dropna(subset=['pv_at_action', 'settled_pv']).copy()

fig = px.scatter(
    plot_df,
    x='pv_at_action',
    y='settled_pv',
    color='action_direction',
    symbol='change_type',
    size='abs_op_delta',
    hover_data=[
        'EpisodeID', 'event_time', 'change_type', 'op_delta',
        'response_time_mins', 'settling_time_mins', 'action_zone',
        'settled_zone', 'movement_towards_limits'
    ],
    title='03LIC_1071 Episode Actions: 03LIC_1071.PV At Action Time Versus Settled Value',
    labels={
        'pv_at_action': '03LIC_1071.PV when action was taken',
        'settled_pv': '03LIC_1071.PV settled around',
        'abs_op_delta': '|action change|',
        'action_direction': 'Action direction',
        'change_type': 'Action type'
    },
    symbol_map={'OP': 'circle', 'SP': 'diamond'}
)

min_axis = min(plot_df['pv_at_action'].min(), plot_df['settled_pv'].min(), TARGET_LOWER_LIMIT) - 1
max_axis = max(plot_df['pv_at_action'].max(), plot_df['settled_pv'].max(), TARGET_UPPER_LIMIT) + 1

fig.add_shape(
    type='rect',
    x0=min_axis, x1=max_axis,
    y0=TARGET_LOWER_LIMIT, y1=TARGET_UPPER_LIMIT,
    fillcolor='lightgreen', opacity=0.15, line_width=0,
    layer='below'
)
fig.add_shape(
    type='rect',
    x0=TARGET_LOWER_LIMIT, x1=TARGET_UPPER_LIMIT,
    y0=min_axis, y1=max_axis,
    fillcolor='lightgreen', opacity=0.15, line_width=0,
    layer='below'
)
fig.add_shape(
    type='line',
    x0=min_axis, y0=min_axis, x1=max_axis, y1=max_axis,
    line=dict(color='gray', dash='dash')
)
fig.add_hline(y=TARGET_LOWER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_hline(y=TARGET_UPPER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_hline(y=71, line_dash='dot', line_color='green')
fig.add_hline(y=28.75, line_dash='dot', line_color='red')
fig.add_vline(x=28.75, line_dash='dot', line_color='red')
fig.add_vline(x=TARGET_LOWER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_vline(x=TARGET_UPPER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_vline(x=71, line_dash='dot', line_color='green')

fig.update_layout(height=650)
fig.show()

In [12]:
# ── Visual 4: 03LIC_1016 action-time PV vs settled PV ───────────────────
plot_df = response_df[response_df['source_tag'] == '03LIC_1016'].dropna(subset=['pv_at_action', 'settled_pv']).copy()

fig = px.scatter(
    plot_df,
    x='pv_at_action',
    y='settled_pv',
    color='action_direction',
    symbol='change_type',
    size='abs_op_delta',
    hover_data=[
        'EpisodeID', 'event_time', 'change_type', 'op_delta',
        'response_time_mins', 'settling_time_mins', 'action_zone',
        'settled_zone', 'movement_towards_limits'
    ],
    title='03LIC_1016 Episode Actions: 03LIC_1071.PV At Action Time Versus Settled Value',
    labels={
        'pv_at_action': '03LIC_1071.PV when action was taken',
        'settled_pv': '03LIC_1071.PV settled around',
        'abs_op_delta': '|action change|',
        'action_direction': 'Action direction',
        'change_type': 'Action type'
    },
    symbol_map={'OP': 'circle', 'SP': 'diamond'}
)

min_axis = min(plot_df['pv_at_action'].min(), plot_df['settled_pv'].min(), TARGET_LOWER_LIMIT) - 1
max_axis = max(plot_df['pv_at_action'].max(), plot_df['settled_pv'].max(), TARGET_UPPER_LIMIT) + 1

fig.add_shape(
    type='rect',
    x0=min_axis, x1=max_axis,
    y0=TARGET_LOWER_LIMIT, y1=TARGET_UPPER_LIMIT,
    fillcolor='lightgreen', opacity=0.15, line_width=0,
    layer='below'
)
fig.add_shape(
    type='rect',
    x0=TARGET_LOWER_LIMIT, x1=TARGET_UPPER_LIMIT,
    y0=min_axis, y1=max_axis,
    fillcolor='lightgreen', opacity=0.15, line_width=0,
    layer='below'
)
fig.add_shape(
    type='line',
    x0=min_axis, y0=min_axis, x1=max_axis, y1=max_axis,
    line=dict(color='gray', dash='dash')
)
fig.add_hline(y=TARGET_LOWER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_hline(y=TARGET_UPPER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_hline(y=71, line_dash='dot', line_color='green')
fig.add_hline(y=28.75, line_dash='dot', line_color='red')
fig.add_vline(x=28.75, line_dash='dot', line_color='red')
fig.add_vline(x=TARGET_LOWER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_vline(x=TARGET_UPPER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_vline(x=71, line_dash='dot', line_color='green')

fig.update_layout(height=650)
fig.show()

In [13]:
# ── Visual 5: 03PIC_1013 action-time PV vs settled PV ───────────────────
plot_df = response_df[response_df['source_tag'] == '03PIC_1013'].dropna(subset=['pv_at_action', 'settled_pv']).copy()

fig = px.scatter(
    plot_df,
    x='pv_at_action',
    y='settled_pv',
    color='action_direction',
    symbol='change_type',
    size='abs_op_delta',
    hover_data=[
        'EpisodeID', 'event_time', 'change_type', 'op_delta',
        'response_time_mins', 'settling_time_mins', 'action_zone',
        'settled_zone', 'movement_towards_limits'
    ],
    title='03PIC_1013 Episode Actions: 03LIC_1071.PV At Action Time Versus Settled Value',
    labels={
        'pv_at_action': '03LIC_1071.PV when action was taken',
        'settled_pv': '03LIC_1071.PV settled around',
        'abs_op_delta': '|action change|',
        'action_direction': 'Action direction',
        'change_type': 'Action type'
    },
    symbol_map={'OP': 'circle', 'SP': 'diamond'}
)

min_axis = min(plot_df['pv_at_action'].min(), plot_df['settled_pv'].min(), TARGET_LOWER_LIMIT) - 1
max_axis = max(plot_df['pv_at_action'].max(), plot_df['settled_pv'].max(), TARGET_UPPER_LIMIT) + 1

fig.add_shape(
    type='rect',
    x0=min_axis, x1=max_axis,
    y0=TARGET_LOWER_LIMIT, y1=TARGET_UPPER_LIMIT,
    fillcolor='lightgreen', opacity=0.15, line_width=0,
    layer='below'
)
fig.add_shape(
    type='rect',
    x0=TARGET_LOWER_LIMIT, x1=TARGET_UPPER_LIMIT,
    y0=min_axis, y1=max_axis,
    fillcolor='lightgreen', opacity=0.15, line_width=0,
    layer='below'
)
fig.add_shape(
    type='line',
    x0=min_axis, y0=min_axis, x1=max_axis, y1=max_axis,
    line=dict(color='gray', dash='dash')
)
fig.add_hline(y=TARGET_LOWER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_hline(y=TARGET_UPPER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_hline(y=71, line_dash='dot', line_color='green')
fig.add_hline(y=28.75, line_dash='dot', line_color='red')
fig.add_vline(x=28.75, line_dash='dot', line_color='red')
fig.add_vline(x=TARGET_LOWER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_vline(x=TARGET_UPPER_LIMIT, line_dash='dot', line_color='firebrick')
fig.add_vline(x=71, line_dash='dot', line_color='green')

fig.update_layout(height=650)
fig.show()

## Extract Successful Control Actions

For each of `03LIC_1071`, `03LIC_1016`, and `03PIC_1013`, extract actions where:
- `03LIC_1071.PV` at action time was **below the lower operating limit**
- The settled PV value is **between the lower and upper operating limits**

These represent control actions that successfully brought the process variable back into the acceptable range. Results are saved to an Excel file with one sheet per tag.

In [20]:
# ── Extract successful control actions per tag and save to Excel ──────────
TARGET_TAGS = ['03LIC_1071', '03LIC_1016', '03PIC_1013']

# Add settled_pv_timestamp: action time + settling_time_mins
response_df['settled_pv_timestamp'] = response_df.apply(
    lambda r: r['event_time_rounded'] + pd.Timedelta(minutes=r['settling_time_mins'])
    if pd.notna(r['settling_time_mins']) else pd.NaT,
    axis=1
)

output_cols = [
    'EpisodeID',
    # 'AlarmStart_rounded_minutes',
    # 'AlarmEnd_rounded_minutes',
    'event_time',
    # 'source_tag',
    'change_type',
    'action_direction',
    'op_delta',
    # 'abs_op_delta',
    'op_value_at_change',
    'pv_at_action',
    'settled_pv',
    'settled_pv_timestamp',
    'pv_change_to_settled',
    # 'response_time_mins',
    'settling_time_mins',
    # 'settling_status',
    # 'available_post_minutes',
    'action_zone',
    'settled_zone',
    'movement_towards_limits',
    'pv_baseline_mean',
    'pv_baseline_std',
    'settled_pv_std',
    'action_rank_in_episode',
    # 'year',
]

output_path = '/home/h604827/ControlActions/RESULTS/successful_control_actions.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for tag in TARGET_TAGS:
        tag_df = response_df[
            (response_df['source_tag'] == tag) &
            (response_df['pv_at_action'] <= TARGET_LOWER_LIMIT) &
            (response_df['settled_pv'] >= TARGET_LOWER_LIMIT) &
            (response_df['settled_pv'] <= TARGET_UPPER_LIMIT)
        ].copy()

        tag_df = tag_df[output_cols].sort_values(
            ['EpisodeID', 'event_time']
        ).reset_index(drop=True)

        sheet_name = tag.replace('03', '').replace('_', '')
        tag_df.to_excel(writer, sheet_name=sheet_name, index=False)

        print(f'{tag}: {len(tag_df)} successful actions across {tag_df["EpisodeID"].nunique()} episodes')
        if len(tag_df) > 0:
            print(f'  Change type: {tag_df["change_type"].value_counts().to_dict()}')
            print(f'  Direction:   {tag_df["action_direction"].value_counts().to_dict()}')
            # print(f'  Median |delta|: {tag_df["abs_op_delta"].median():.2f}')
            print(f'  Median settling time: {tag_df["settling_time_mins"].median():.1f} min')
            print(f'  Median PV recovery: {tag_df["pv_change_to_settled"].median():.2f}')
        print()

print(f'Saved to {output_path}')

03LIC_1071: 11 successful actions across 11 episodes
  Change type: {'SP': 11}
  Direction:   {'increase': 11}
  Median settling time: 9.0 min
  Median PV recovery: 13.28

03LIC_1016: 3 successful actions across 3 episodes
  Change type: {'SP': 3}
  Direction:   {'increase': 3}
  Median settling time: 17.0 min
  Median PV recovery: 9.61

03PIC_1013: 13 successful actions across 13 episodes
  Change type: {'OP': 13}
  Direction:   {'decrease': 9, 'increase': 4}
  Median settling time: 12.0 min
  Median PV recovery: 6.17

Saved to /home/h604827/ControlActions/RESULTS/successful_control_actions.xlsx


In [16]:
# Distribution of 03LIC_1071 actions among OP vs SP (during alarm duration only)
actions_1071_alarm = episode_actions[
    (episode_actions['Source'] == '03LIC_1071') &
    (episode_actions['VT_Start'] >= episode_actions['AlarmStart_rounded_minutes']) &
    (episode_actions['VT_Start'] <= episode_actions['AlarmEnd_rounded_minutes'])
].copy()

actions_1071_alarm['change_type'] = actions_1071_alarm['Description']
actions_1071_alarm['action_direction'] = np.where(actions_1071_alarm['delta'] > 0, 'increase', 'decrease')

dist_1071 = (
    actions_1071_alarm['change_type']
    .value_counts()
    .rename_axis('change_type')
    .reset_index(name='n_actions')
)
dist_1071['pct'] = 100 * dist_1071['n_actions'] / dist_1071['n_actions'].sum()

print('03LIC_1071 action distribution during alarm duration (OP vs SP):')
display(dist_1071)

# Direction split within OP/SP during alarm duration
direction_split_1071 = (
    actions_1071_alarm
    .groupby(['change_type', 'action_direction'])
    .size()
    .reset_index(name='n_actions')
    .sort_values(['change_type', 'action_direction'])
)
display(direction_split_1071)

if len(dist_1071) > 0:
    fig = px.pie(
        dist_1071,
        names='change_type',
        values='n_actions',
        hole=0.45,
        title='03LIC_1071 Actions During Alarm Duration: OP vs SP'
    )
    fig.update_traces(textposition='inside', textinfo='percent+label')
    fig.show()
else:
    print('No 03LIC_1071 actions found within alarm durations.')

03LIC_1071 action distribution during alarm duration (OP vs SP):


,change_type,n_actions,pct
0,OP,493,73.472429
1,SP,178,26.527571


,change_type,action_direction,n_actions
0,OP,decrease,220
1,OP,increase,273
2,SP,decrease,68
3,SP,increase,110
